# MIT φ⁴ flow port: kappa/lambda + inverse-blocking condition

This notebook is restricted to the scalar finite-λ φ⁴ theory in our normalization. It uses MIT-style RealNVP affine couplings, but inputs are `(kappa, lambda)`, not `(M2, lambda_MIT)`. The conditional variant keeps even-even sites fixed after inverse-kernel momentum upscaling.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import torch
import numpy as np

from invblock_mit_nf.actions import Phi4Params, Phi4Action, MITPhi4Action, rescale_ours_to_mit
from invblock_mit_nf.blocking import BlockingKernel2D, momentum_inverse_upscale_to_even_even
from invblock_mit_nf.conditional_flow import ConditionalPhi4Flow
from invblock_mit_nf.train_inverse_kl import reverse_kl_step

## Parameters in our normalization

MIT tutorial point check: `M2=-4, lambda_MIT=8` corresponds to `lambda=0.5, kappa=0.25`.

In [ ]:
kappa = 0.25
lam = 0.5
params = Phi4Params(kappa=kappa, lam=lam)
M2, lam_mit = params.to_mit()
print({"kappa": kappa, "lambda": lam, "MIT_M2": M2, "MIT_lambda": lam_mit})

## Action cross-check
The MIT action and our action agree after the field rescaling `varphi=sqrt(kappa)*phi`, up to the additive constant `lambda * volume`.

In [ ]:
L = 16
x_ours = torch.randn(8, L, L)
S_ours = Phi4Action(params)(x_ours)
S_mit = MITPhi4Action(M2, lam_mit)(rescale_ours_to_mit(x_ours, kappa))
print(torch.max(torch.abs(S_mit - (S_ours - lam * L * L))).item())

## Coarse-to-condition uplift
Replace the identity kernel below by the finite-λ perfect-blocking 5x5 kernel stencil.

In [ ]:
coarse = torch.randn(16, 8, 8)
kernel = BlockingKernel2D({(0, 0): 1.0}, name="identity_placeholder")
condition = momentum_inverse_upscale_to_even_even(coarse, kernel, Lf=16)
print(condition.shape, torch.max(torch.abs(condition[:, 0::2, 0::2] - coarse)).item())

## Conditional flow smoke step
This is the inverse-KL objective `E_q[log q + S_fine]`. Even-even sites are fixed exactly.

In [ ]:
flow = ConditionalPhi4Flow(L=16, n_layers=4, hidden=16)
action = Phi4Action(params)
opt = torch.optim.Adam(flow.parameters(), lr=1e-3)
metrics = reverse_kl_step(flow, action, opt, condition[:8])
y, logq = flow.sample(8, condition[:8])
print(metrics)
print("max fixed-site error", torch.max(torch.abs(y[:,0::2,0::2] - condition[:8,0::2,0::2])).item())

## Next replacements

1. Load actual 8² coarse reference configurations instead of Gaussian dummy `coarse`.
2. Load the finite-λ 5x5 perfect-blocking kernel.
3. Train with checkpointing and a metrics CSV.
4. Compare generated 16² observables to direct 16² reference, not only ESS.